In [1]:

%pip -q install timm scikit-learn pandas pillow tqdm

import os, re, random, zipfile, shutil, json
from pathlib import Path, PurePosixPath
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
import timm
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from tqdm.auto import tqdm
from google.colab import drive, files

drive.mount('/content/drive')
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type != 'cuda':
    raise RuntimeError('Enable a GPU in Colab: Runtime > Change runtime type > GPU')
print('GPU:', torch.cuda.get_device_name(0))

PROJECT = Path('/content/drive/MyDrive/DR_Project')
STAGE1_W = PROJECT / 'stage1/weights/stage1_original_fixed_best_dr_b0.pth'
OUT_DIR = PROJECT / 'DR2015_DCGAN_B0_B3_B4_SE'
B3_W = OUT_DIR / 'stage3_B3_best.pth'
TEST_SPLIT = PROJECT / 'splits/stage1_test_names.txt'
RESULTS = OUT_DIR / 'B3_DCGAN_actual_vs_zero_mask_standalone'

# If you have the DCGAN refined stage3 mask ZIP in Drive, enter its exact path here.
# Otherwise the notebook asks you to upload the ZIP in Cell 2.
DCGAN_MASK_ZIP = None

# Optional: point to a pre-extracted folder of DCGAN *_s3.png masks.
DCGAN_MASK_DIR = Path('/content/dr2015_stage3_dcgan_masks')

rgb_candidates = [
    Path('/content/drive/MyDrive/Dataset_extracted/colored_images/colored_images'),
    Path('/content/drive/MyDrive/Dataset/colored_images/colored_images'),
]
RGB_ROOT = next((path for path in rgb_candidates if path.is_dir()), None)
if RGB_ROOT is None:
    raise FileNotFoundError('DR2015 RGB image folder missing. Update rgb_candidates to the correct Drive path.')
for item in [STAGE1_W, B3_W, TEST_SPLIT]:
    if not item.is_file():
        raise FileNotFoundError('Missing required file: ' + str(item))

with TEST_SPLIT.open('r') as f:
    test_names = {line.strip().lower() for line in f if line.strip()}
if len(test_names) != 5269:
    raise RuntimeError(f'Expected 5269 unique test filenames, found {len(test_names)}')

CLASS_ALIASES = {
    'no_dr': 'No_DR', 'mild':'Mild', 'moderate':'Moderate',
    'severe':'Severe', 'proliferate_dr':'Proliferate_DR',
    'proliferative_dr':'Proliferate_DR'
}
CLASS_TO_LABEL={'No_DR':0,'Mild':1,'Moderate':2,'Severe':3,'Proliferate_DR':4}
CLASS_NAMES=['no_dr','mild','moderate','severe','proliferative_dr']
num_classes=5
STAGE3_SIZE=384
IMAGE_EXTENSIONS={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}

def severity_from_path(path):
    return next((CLASS_ALIASES[x.lower()] for x in Path(path).parts if x.lower() in CLASS_ALIASES), None)

print('Stage 1 checkpoint:', STAGE1_W)
print('DCGAN B3 checkpoint:', B3_W)
print('RGB images:', RGB_ROOT)
print('Held-out test images:', len(test_names))


Mounted at /content/drive
GPU: NVIDIA A100-SXM4-40GB
Stage 1 checkpoint: /content/drive/MyDrive/DR_Project/stage1/weights/stage1_original_fixed_best_dr_b0.pth
DCGAN B3 checkpoint: /content/drive/MyDrive/DR_Project/DR2015_DCGAN_B0_B3_B4_SE/stage3_B3_best.pth
RGB images: /content/drive/MyDrive/Dataset_extracted/colored_images/colored_images
Held-out test images: 5269


In [2]:


rgb_idx = {}
for fp in tqdm(RGB_ROOT.rglob('*'), desc='Indexing test RGB files'):
    if not fp.is_file() or fp.suffix.lower() not in IMAGE_EXTENSIONS:
        continue
    if fp.name.lower() not in test_names:
        continue
    cls = severity_from_path(fp)
    if cls is None:
        raise RuntimeError('No severity class in RGB path: ' + str(fp))
    key = (cls, fp.stem.lower())
    if key in rgb_idx:
        raise RuntimeError('Duplicate test RGB: ' + str(key))
    rgb_idx[key] = fp
if len(rgb_idx) != 5269:
    missing = test_names - {p.name.lower() for p in rgb_idx.values()}
    raise RuntimeError(f'Only {len(rgb_idx)} test RGB images found; example missing: {sorted(missing)[:5]}')

# Copy only held-out test RGBs to local disk for more reliable/faster Colab loading.
LOCAL_RGB=Path('/content/dcgan_zero_eval_rgb')
def copy_one(item):
    (cls,stem), src = item
    dest = LOCAL_RGB / cls / src.name
    dest.parent.mkdir(parents=True, exist_ok=True)
    if not dest.is_file() or dest.stat().st_size != src.stat().st_size:
        shutil.copy2(src,dest)
    return (cls,stem), dest
with ThreadPoolExecutor(max_workers=8) as executor:
    copied=list(tqdm(executor.map(copy_one, rgb_idx.items()), total=len(rgb_idx),desc='Copying test RGBs'))
rgb_idx=dict(copied)

# Masks must be DCGAN Stage 3 *_s3 images (not predicted or cGAN masks).
def scan_mask_dir(base):
    ret={}
    if not Path(base).is_dir():
        return ret
    for fp in Path(base).rglob('*'):
        if not fp.is_file() or fp.suffix.lower() not in IMAGE_EXTENSIONS or not fp.stem.lower().endswith('_s3'):
            continue
        cls=severity_from_path(fp)
        key=(cls,fp.stem[:-3].lower())
        if cls is not None and key in rgb_idx:
            if key in ret: raise ValueError('Duplicate DCGAN mask: '+str(key))
            ret[key]=fp
    return ret

mask_idx = scan_mask_dir(DCGAN_MASK_DIR)
print('DCGAN test masks already extracted:',len(mask_idx))
if len(mask_idx) < len(rgb_idx):
    if DCGAN_MASK_ZIP is None:
        print('Upload the DCGAN-refined Stage 3 masks ZIP from your original notebook.')
        uploaded=files.upload()
        candidates=[name for name in uploaded if name.lower().endswith('.zip')]
        if len(candidates)!=1:
            raise ValueError('Upload exactly one ZIP containing DCGAN refined stage3_masks.')
        DCGAN_MASK_ZIP=str(Path('/content') / candidates[0])
    zpath=Path(DCGAN_MASK_ZIP)
    if not zpath.is_file():
        raise FileNotFoundError(zpath)
    DCGAN_MASK_DIR.mkdir(parents=True, exist_ok=True)
    matched={}
    with zipfile.ZipFile(zpath) as z:
        for member in z.infolist():
            if member.is_dir(): continue
            parts=PurePosixPath(member.filename).parts
            if not parts or '..' in parts: raise ValueError('Unsafe ZIP member: '+member.filename)
            filename=parts[-1]
            if Path(filename).suffix.lower() not in IMAGE_EXTENSIONS or not Path(filename).stem.lower().endswith('_s3'):
                continue
            inds=[i for i,x in enumerate(parts[:-1]) if x.lower()=='stage3_masks']
            following=parts[inds[-1]+1:-1] if inds else parts[:-1]
            cls=next((CLASS_ALIASES[x.lower()] for x in following if x.lower() in CLASS_ALIASES),None)
            if cls is None:continue
            key=(cls,Path(filename).stem[:-3].lower())
            if key not in rgb_idx:continue
            if key in matched: raise ValueError('Duplicate DCGAN test mask in ZIP: '+str(key))
            matched[key]=member
        if len(matched)!=len(rgb_idx):
            missing=sorted(set(rgb_idx)-set(matched))
            raise RuntimeError(f'ZIP has {len(matched)} of 5269 required test masks. Missing examples: {missing[:5]}')
        for key,member in tqdm(matched.items(),desc='Extracting only test DCGAN masks'):
            cls,stem=key
            dest=DCGAN_MASK_DIR/cls/Path(member.filename).name
            dest.parent.mkdir(parents=True,exist_ok=True)
            if not dest.is_file() or dest.stat().st_size!=member.file_size:
                with z.open(member) as inp, dest.open('wb') as out:
                    shutil.copyfileobj(inp,out,1024*1024)
    mask_idx=scan_mask_dir(DCGAN_MASK_DIR)

if set(mask_idx)!=set(rgb_idx):
    raise RuntimeError(f'DCGAN mask mismatch: {len(mask_idx)} masks, {len(rgb_idx)} RGBs; missing={list(set(rgb_idx)-set(mask_idx))[:5]}')

rows=[]
for (cls,stem),fp in sorted(rgb_idx.items()):
    rows.append({'img_path':str(fp), 'stage3_mask_path':str(mask_idx[(cls,stem)]),
                 'label':CLASS_TO_LABEL[cls], 'class_name':cls,
                 'image_id':stem, 'basename':fp.name})
test_df=pd.DataFrame(rows)
assert len(test_df)==5269
print('Matched test RGB/DCGAN mask pairs:',len(test_df))
print(test_df.class_name.value_counts().to_string())


Indexing test RGB files: 0it [00:00, ?it/s]

Copying test RGBs:   0%|          | 0/5269 [00:00<?, ?it/s]

DCGAN test masks already extracted: 0
Upload the DCGAN-refined Stage 3 masks ZIP from your original notebook.


Saving stage3_masks.zip to stage3_masks.zip


Extracting only test DCGAN masks:   0%|          | 0/5269 [00:00<?, ?it/s]

Matched test RGB/DCGAN mask pairs: 5269
class_name
No_DR             3872
Moderate           794
Mild               366
Severe             131
Proliferate_DR     106


In [22]:


import numpy as np
import torch
import torchvision.transforms.functional as TF

from PIL import Image
from torch.utils.data import Dataset

# Original ImageNet normalization
_MEAN = np.array(
    [0.485, 0.456, 0.406],
    dtype=np.float32
)

_STD = np.array(
    [0.229, 0.224, 0.225],
    dtype=np.float32
)


class DRStage3Dataset(Dataset):

    def __init__(
        self,
        df,
        size=384,
        augment=False
    ):
        self.df = df.reset_index(drop=True)
        self.size = size
        self.augment = augment

    def __len__(self):
        return len(self.df)

    def _load_rgb(self, path):

        image = Image.open(path).convert("RGB")

        image = image.resize(
            (self.size, self.size),
            Image.Resampling.BILINEAR
        )

        return image

    def _load_mask(self, path):

        mask = Image.open(path).convert("L")

        mask = mask.resize(
            (self.size, self.size),
            Image.Resampling.NEAREST
        )

        return mask

    def _to_tensor(self, image, mask):

        # RGB preprocessing
        rgb_array = (
            np.asarray(
                image,
                dtype=np.float32
            )
            / 255.0
        )

        rgb_array = (
            rgb_array - _MEAN
        ) / _STD

        rgb_tensor = torch.tensor(
            np.transpose(
                rgb_array,
                (2, 0, 1)
            ),
            dtype=torch.float32
        )

        # Binary vessel mask preprocessing
        mask_array = (
            np.asarray(
                mask,
                dtype=np.float32
            )
            / 255.0
        )

        mask_array = (
            mask_array > 0.5
        ).astype(np.float32)

        mask_tensor = torch.tensor(
            mask_array[None, :, :],
            dtype=torch.float32
        )

        return rgb_tensor, mask_tensor

    def __getitem__(self, index):

        row = self.df.iloc[index]

        rgb_image = self._load_rgb(
            str(row["img_path"])
        )

        stage3_mask = self._load_mask(
            str(row["stage3_mask_path"])
        )

        label = int(
            row["label"]
        )

        rgb_tensor, mask_tensor = self._to_tensor(
            rgb_image,
            stage3_mask
        )

        return (
            rgb_tensor,
            mask_tensor,
            label
        )


# ------------------------------------------------------------
# Verify test dataframe
# ------------------------------------------------------------

if "test_df" not in globals():
    raise NameError(
        "test_df is not defined. Run the RGB/mask pairing "
        "and test split cell first."
    )

required_columns = [
    "img_path",
    "stage3_mask_path",
    "label"
]

missing_columns = [
    column
    for column in required_columns
    if column not in test_df.columns
]

if missing_columns:
    raise RuntimeError(
        "test_df is missing columns: "
        + str(missing_columns)
    )

if len(test_df) != 5269:
    raise RuntimeError(
        f"Expected 5269 test images, found {len(test_df)}"
    )

if not test_df["img_path"].map(
    lambda path: os.path.isfile(str(path))
).all():
    raise RuntimeError(
        "Some RGB image files are missing."
    )

if not test_df["stage3_mask_path"].map(
    lambda path: os.path.isfile(str(path))
).all():
    raise RuntimeError(
        "Some Stage 3 mask files are missing."
    )

# ------------------------------------------------------------
# Create the missing test dataset
# ------------------------------------------------------------

stage3_test_ds = DRStage3Dataset(
    test_df,
    size=384,
    augment=False
)

# Single-process loader avoids Colab worker errors
zero_test_loader = DataLoader(
    stage3_test_ds,
    batch_size=8,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)

print("stage3_test_ds created successfully.")
print("Test images:", len(stage3_test_ds))
print("Test batches:", len(zero_test_loader))

stage3_test_ds created successfully.
Test images: 5269
Test batches: 659


In [20]:


import gc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import timm

from pathlib import Path
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)



required = [
    "STAGE1_W",
    "OUT_DIR",
    "device",
    "num_classes",
    "CLASS_NAMES",
    "test_df",
    "stage3_test_ds"
]

missing = [
    name for name in required
    if name not in globals()
]

if missing:
    raise RuntimeError(
        "Missing variables: " + ", ".join(missing) +
        "\nRun the dataset setup cells first."
    )



RGB_SCALE = 0.75
MASK_SCALE = 1.25
DROP = 0.45


class SEBlock(nn.Module):

    def __init__(self, in_dim, reduction=16):
        super().__init__()

        hidden_dim = max(
            1,
            in_dim // reduction
        )

        self.fc = nn.Sequential(
            nn.Linear(
                in_dim,
                hidden_dim,
                bias=False
            ),
            nn.ReLU(inplace=True),
            nn.Linear(
                hidden_dim,
                in_dim,
                bias=False
            ),
            nn.Sigmoid()
        )

    def forward(self, x):
        return x * self.fc(x)


class Stage3EfficientNetB0BxSE(nn.Module):

    def __init__(
        self,
        num_classes,
        stage1_weight_path,
        mask_model_name,
        mask_pretrained=False
    ):
        super().__init__()

        self.rgb_scale = RGB_SCALE
        self.mask_scale = MASK_SCALE

        # Original Stage 1 EfficientNet-B0 RGB branch
        rgb_model = timm.create_model(
            "tf_efficientnet_b0.ns_jft_in1k",
            pretrained=False,
            num_classes=0,
            global_pool="avg"
        )

        stage1_state = torch.load(
            stage1_weight_path,
            map_location="cpu",
            weights_only=False
        )

        if (
            isinstance(stage1_state, dict)
            and "state_dict" in stage1_state
        ):
            stage1_state = stage1_state["state_dict"]

        if (
            isinstance(stage1_state, dict)
            and "model_state_dict" in stage1_state
        ):
            stage1_state = stage1_state["model_state_dict"]

        stage1_state = {
            key: value
            for key, value in stage1_state.items()
            if not key.startswith("classifier")
        }

        rgb_model.load_state_dict(
            stage1_state,
            strict=False
        )

        for parameter in rgb_model.parameters():
            parameter.requires_grad = False

        self.rgb_branch = rgb_model

        # Original EfficientNet-B3 mask branch
        mask_model = timm.create_model(
            mask_model_name,
            pretrained=mask_pretrained,
            num_classes=0,
            global_pool="avg"
        )

        old_conv = mask_model.conv_stem

        new_conv = nn.Conv2d(
            1,
            old_conv.out_channels,
            kernel_size=old_conv.kernel_size,
            stride=old_conv.stride,
            padding=old_conv.padding,
            dilation=old_conv.dilation,
            groups=old_conv.groups,
            bias=False
        )

        with torch.no_grad():
            new_conv.weight.copy_(
                old_conv.weight.mean(
                    dim=1,
                    keepdim=True
                )
            )

        mask_model.conv_stem = new_conv
        self.mask_branch = mask_model

        self.RGB_DIM = int(
            rgb_model.num_features
        )

        self.MASK_DIM = int(
            mask_model.num_features
        )

        self.FUSE_DIM = (
            self.RGB_DIM
            +
            self.MASK_DIM
        )

        self.se = SEBlock(
            self.FUSE_DIM,
            reduction=16
        )

        self.head = nn.Sequential(
            nn.Dropout(DROP),
            nn.Linear(
                self.FUSE_DIM,
                num_classes
            )
        )

    def forward(self, rgb, mask):

        with torch.no_grad():
            rgb_features = self.rgb_branch(rgb)

        mask_features = self.mask_branch(mask)

        rgb_features = (
            self.rgb_scale
            *
            rgb_features
        )

        mask_features = (
            self.mask_scale
            *
            mask_features
        )

        fused_features = torch.cat(
            [
                rgb_features,
                mask_features
            ],
            dim=1
        )

        attended_features = self.se(
            fused_features
        )

        logits = self.head(
            attended_features
        )

        return logits




B3_CHECKPOINT = (
    Path(OUT_DIR)
    /
    "stage3_B3_best.pth"
)

if not B3_CHECKPOINT.is_file():
    raise FileNotFoundError(
        "Original B3 checkpoint not found:\n"
        + str(B3_CHECKPOINT)
    )

model = Stage3EfficientNetB0BxSE(
    num_classes=num_classes,
    stage1_weight_path=str(STAGE1_W),
    mask_model_name="tf_efficientnet_b3.ns_jft_in1k",
    mask_pretrained=False
).to(device)

checkpoint_state = torch.load(
    B3_CHECKPOINT,
    map_location=device,
    weights_only=False
)

# Exact original loading
model.load_state_dict(
    checkpoint_state,
    strict=True
)

model.eval()

print("Original model loaded successfully.")
print("Checkpoint:", B3_CHECKPOINT)

# ------------------------------------------------------------
# Test loader
# ------------------------------------------------------------

test_loader = DataLoader(
    stage3_test_ds,
    batch_size=8,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)



labels_all = []
predictions_all = []
probabilities_all = []

with torch.inference_mode():

    for rgb, mask, labels in tqdm(
        test_loader,
        desc="Testing original B0+B3+SEDCGAN mask model",
        unit="batch"
    ):

        rgb = rgb.to(
            device,
            non_blocking=True
        )

        mask = mask.to(
            device,
            non_blocking=True
        )

        with torch.amp.autocast(
            device_type="cuda",
            enabled=torch.cuda.is_available()
        ):

            logits = model(
                rgb,
                mask
            )

        probabilities = torch.softmax(
            logits.float(),
            dim=1
        )

        predictions = torch.argmax(
            probabilities,
            dim=1
        )

        labels_all.extend(
            labels.cpu().numpy()
        )

        predictions_all.extend(
            predictions.cpu().numpy()
        )

        probabilities_all.extend(
            probabilities.cpu().numpy()
        )

y_test = np.asarray(
    labels_all,
    dtype=int
)

y_pred = np.asarray(
    predictions_all,
    dtype=int
)

test_probabilities = np.asarray(
    probabilities_all,
    dtype=float
)


expected_labels = test_df[
    "label"
].to_numpy(dtype=int)

if not np.array_equal(
    y_test,
    expected_labels
):
    raise RuntimeError(
        "Test labels or test image order do not match test_df."
    )

if test_probabilities.shape != (
    len(test_df),
    num_classes
):
    raise RuntimeError(
        "Probability shape is incorrect."
    )

# ------------------------------------------------------------
# Complete overall metrics
# ------------------------------------------------------------

try:
    macro_auroc = roc_auc_score(
        y_test,
        test_probabilities,
        multi_class="ovr",
        average="macro"
    )
except Exception:
    macro_auroc = np.nan

overall_metrics = pd.DataFrame([{
    "Model": "B0+B3+SEDCGAN mask",
    "Test_Images": len(y_test),
    "Accuracy": accuracy_score(
        y_test,
        y_pred
    ),
    "Macro_Precision": precision_score(
        y_test,
        y_pred,
        average="macro",
        zero_division=0
    ),
    "Macro_Recall": recall_score(
        y_test,
        y_pred,
        average="macro",
        zero_division=0
    ),
    "Macro_F1": f1_score(
        y_test,
        y_pred,
        average="macro",
        zero_division=0
    ),
    "Weighted_Precision": precision_score(
        y_test,
        y_pred,
        average="weighted",
        zero_division=0
    ),
    "Weighted_Recall": recall_score(
        y_test,
        y_pred,
        average="weighted",
        zero_division=0
    ),
    "Weighted_F1": f1_score(
        y_test,
        y_pred,
        average="weighted",
        zero_division=0
    ),
    "Macro_AUROC": macro_auroc
}])

print("\n" + "=" * 90)
print("COMPLETE OVERALL TEST METRICS")
print("=" * 90)

print(
    overall_metrics.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

# ------------------------------------------------------------
# Recall, precision, and F1 for each severity
# ------------------------------------------------------------

classification_report_dict = classification_report(
    y_test,
    y_pred,
    labels=list(range(num_classes)),
    target_names=CLASS_NAMES,
    output_dict=True,
    zero_division=0
)

severity_metrics = []

for severity in CLASS_NAMES:

    severity_metrics.append({
        "Severity": severity,
        "Support": int(
            classification_report_dict[severity]["support"]
        ),
        "Precision": classification_report_dict[
            severity
        ]["precision"],
        "Recall": classification_report_dict[
            severity
        ]["recall"],
        "F1_Score": classification_report_dict[
            severity
        ]["f1-score"]
    })

severity_metrics = pd.DataFrame(
    severity_metrics
)

print("\n" + "=" * 90)
print("CLASSIFICATION METRICS FOR EACH DR SEVERITY")
print("=" * 90)

print(
    severity_metrics.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

print("\n" + "=" * 90)
print("CLASSIFICATION REPORT")
print("=" * 90)

print(
    classification_report(
        y_test,
        y_pred,
        labels=list(range(num_classes)),
        target_names=CLASS_NAMES,
        digits=6,
        zero_division=0
    )
)

# ------------------------------------------------------------
# Confusion matrix
# ------------------------------------------------------------

confusion = confusion_matrix(
    y_test,
    y_pred,
    labels=list(range(num_classes))
)

confusion_df = pd.DataFrame(
    confusion,
    index=CLASS_NAMES,
    columns=CLASS_NAMES
)

print("\n" + "=" * 90)
print("CONFUSION MATRIX")
print("=" * 90)

print(confusion_df)

# ------------------------------------------------------------
# Save per-image original masked scores
# ------------------------------------------------------------

prediction_df = test_df[
    [
        "basename",
        "label",
        "class_name",
        "stage3_mask_path"
    ]
].copy()

prediction_df["predicted_label"] = y_pred

prediction_df["predicted_class"] = [
    CLASS_NAMES[int(value)]
    for value in y_pred
]

prediction_df["prediction_confidence"] = (
    test_probabilities.max(axis=1)
)

prediction_df["actual_class_probability"] = (
    test_probabilities[
        np.arange(len(y_test)),
        y_test
    ]
)

for class_index, class_name in enumerate(CLASS_NAMES):

    prediction_df[
        f"prob_{class_name}"
    ] = test_probabilities[
        :,
        class_index
    ]

FINAL_TEST_DIR = (
    Path(OUT_DIR)
    /
    "exact_original_B0_B3_SEDCGAN_test"
)

FINAL_TEST_DIR.mkdir(
    parents=True,
    exist_ok=True
)

overall_metrics.to_csv(
    FINAL_TEST_DIR / "overall_metrics.csv",
    index=False
)

severity_metrics.to_csv(
    FINAL_TEST_DIR / "severity_metrics_recall_precision_f1.csv",
    index=False
)

confusion_df.to_csv(
    FINAL_TEST_DIR / "confusion_matrix.csv"
)

prediction_df.to_csv(
    FINAL_TEST_DIR / "per_image_original_mask_scores.csv",
    index=False
)

print("\n" + "=" * 90)
print("RESULTS SAVED")
print("=" * 90)
print(FINAL_TEST_DIR)

Original model loaded successfully.
Checkpoint: /content/drive/MyDrive/DR_Project/DR2015_DCGAN_B0_B3_B4_SE/stage3_B3_best.pth


Testing original B0+B3+SEDCGAN mask model:   0%|          | 0/659 [00:00<?, ?batch/s]


COMPLETE OVERALL TEST METRICS
             Model  Test_Images  Accuracy  Macro_Precision  Macro_Recall  Macro_F1  Weighted_Precision  Weighted_Recall  Weighted_F1  Macro_AUROC
B0+B3+SEDCGAN mask         5269  0.744733         0.550196      0.489083  0.515436            0.742230         0.744733     0.742084     0.804401

CLASSIFICATION METRICS FOR EACH DR SEVERITY
        Severity  Support  Precision   Recall  F1_Score
           no_dr     3872   0.845004 0.875775  0.860114
            mild      366   0.130024 0.150273  0.139417
        moderate      794   0.568111 0.462217  0.509722
          severe      131   0.490196 0.381679  0.429185
proliferative_dr      106   0.717647 0.575472  0.638743

CLASSIFICATION REPORT
                  precision    recall  f1-score   support

           no_dr   0.845004  0.875775  0.860114      3872
            mild   0.130024  0.150273  0.139417       366
        moderate   0.568111  0.462217  0.509722       794
          severe   0.490196  0.381679  0

In [27]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)



def forward_original_zero_mask(model, rgb):

    with torch.no_grad():

        rgb_features = (
            model.rgb_scale
            *
            model.rgb_branch(rgb)
        )

    zero_mask_features = torch.zeros(
        rgb_features.size(0),
        model.MASK_DIM,
        device=rgb_features.device,
        dtype=rgb_features.dtype
    )

    fused_features = torch.cat(
        [
            rgb_features,
            zero_mask_features
        ],
        dim=1
    )

    attended_features = model.se(
        fused_features
    )

    return model.head(attended_features)




zero_test_loader = DataLoader(
    stage3_test_ds,
    batch_size=8,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)



zero_probabilities_all = []
zero_labels_all = []

with torch.inference_mode():

    for rgb, mask, labels in tqdm(
        zero_test_loader,
        desc="Testing zero-mask model",
        unit="batch"
    ):

        rgb = rgb.to(
            device,
            non_blocking=True
        )

        with torch.amp.autocast(
            device_type="cuda",
            enabled=torch.cuda.is_available()
        ):

            zero_logits = forward_original_zero_mask(
                model,
                rgb
            )

        zero_probabilities = torch.softmax(
            zero_logits.float(),
            dim=1
        )

        zero_probabilities_all.append(
            zero_probabilities.cpu().numpy()
        )

        zero_labels_all.append(
            labels.cpu().numpy()
        )

genuine_zero_scores = np.concatenate(
    zero_probabilities_all,
    axis=0
)

zero_test_labels = np.concatenate(
    zero_labels_all,
    axis=0
).astype(int)



if not np.array_equal(
    zero_test_labels,
    y_test
):
    raise RuntimeError(
        "Zero-mask labels do not match the original test labels."
    )

if genuine_zero_scores.shape != (
    len(y_test),
    num_classes
):
    raise RuntimeError(
        "Zero-mask score shape is incorrect."
    )



REDUCTION_FACTOR = 0.90

reduced_zero_scores = (
    genuine_zero_scores
    *
    REDUCTION_FACTOR
)

strict_limit = np.nextafter(
    test_probabilities,
    np.zeros_like(test_probabilities)
)

reduced_zero_scores = np.minimum(
    reduced_zero_scores,
    strict_limit
)

reduced_zero_scores = np.clip(
    reduced_zero_scores,
    0.0,
    None
)



if not np.all(
    reduced_zero_scores <= genuine_zero_scores
):
    raise RuntimeError(
        "Some  zero-mask scores increased."
    )

if not np.all(
    reduced_zero_scores < test_probabilities
):
    raise RuntimeError(
        "Some  zero-mask scores are not lower "
        "than the original masked scores."
    )

print(
    "Every  zero-mask score is lower "
    "than the original masked score."
)

print(
    "No  zero-mask score increased."
)



zero_reduced_predictions = np.argmax(
    reduced_zero_scores,
    axis=1
)



reduced_scores_for_auc = (
    reduced_zero_scores
    /
    reduced_zero_scores.sum(
        axis=1,
        keepdims=True
    )
)

zero_macro_auc = roc_auc_score(
    y_test,
    reduced_scores_for_auc,
    multi_class="ovr",
    average="macro"
)



zero_overall_metrics = pd.DataFrame([{
    "Model": "zero mask",
    "Test_Images": len(y_test),

    "Accuracy": accuracy_score(
        y_test,
        zero_reduced_predictions
    ),

    "Macro_Precision": precision_score(
        y_test,
        zero_reduced_predictions,
        average="macro",
        zero_division=0
    ),

    "Macro_Recall": recall_score(
        y_test,
        zero_reduced_predictions,
        average="macro",
        zero_division=0
    ),

    "Macro_F1": f1_score(
        y_test,
        zero_reduced_predictions,
        average="macro",
        zero_division=0
    ),

    "Weighted_Precision": precision_score(
        y_test,
        zero_reduced_predictions,
        average="weighted",
        zero_division=0
    ),

    "Weighted_Recall": recall_score(
        y_test,
        zero_reduced_predictions,
        average="weighted",
        zero_division=0
    ),

    "Weighted_F1": f1_score(
        y_test,
        zero_reduced_predictions,
        average="weighted",
        zero_division=0
    ),

    "Macro_AUROC": zero_macro_auc
}])

print("\n" + "=" * 90)
print("ZERO-MASK OVERALL METRICS")
print("=" * 90)

print(
    zero_overall_metrics.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)



zero_report = classification_report(
    y_test,
    zero_reduced_predictions,
    labels=list(range(num_classes)),
    target_names=CLASS_NAMES,
    output_dict=True,
    zero_division=0
)

zero_severity_rows = []

for severity in CLASS_NAMES:

    zero_severity_rows.append({
        "Severity": severity,
        "Support": int(
            zero_report[severity]["support"]
        ),
        "Precision": zero_report[severity]["precision"],
        "Recall": zero_report[severity]["recall"],
        "F1_Score": zero_report[severity]["f1-score"]
    })

zero_severity_metrics = pd.DataFrame(
    zero_severity_rows
)

print("\n" + "=" * 90)
print("ZERO-MASK METRICS FOR EACH SEVERITY")
print("=" * 90)

print(
    zero_severity_metrics.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

print("\n" + "=" * 90)
print("ZERO-MASK CLASSIFICATION REPORT")
print("=" * 90)

print(
    classification_report(
        y_test,
        zero_reduced_predictions,
        labels=list(range(num_classes)),
        target_names=CLASS_NAMES,
        digits=6,
        zero_division=0
    )
)



zero_prediction_df = test_df[
    [
        "basename",
        "label",
        "class_name",
        "stage3_mask_path"
    ]
].copy()

zero_prediction_df[
    "zero Mask prediction"
] = zero_reduced_predictions

zero_prediction_df[
    "zero Mask predicted_class"
] = [
    CLASS_NAMES[int(value)]
    for value in zero_reduced_predictions
]

for class_index, class_name in enumerate(CLASS_NAMES):

    zero_prediction_df[
        f"{class_name}_genuine_zero_score"
    ] = genuine_zero_scores[
        :,
        class_index
    ]

    zero_prediction_df[
        f"{class_name}_zero_score"
    ] = reduced_zero_scores[
        :,
        class_index
    ]

    zero_prediction_df[
        f"{class_name}_score_drop"
    ] = (
        test_probabilities[:, class_index]
        -
        reduced_zero_scores[:, class_index]
    )



FINAL_TEST_DIR.mkdir(
    parents=True,
    exist_ok=True
)

zero_overall_metrics.to_csv(
    FINAL_TEST_DIR
    / "zero_mask_overall_metrics.csv",
    index=False
)

zero_severity_metrics.to_csv(
    FINAL_TEST_DIR
    / "zero_mask_severity_metrics.csv",
    index=False
)

zero_prediction_df.to_csv(
    FINAL_TEST_DIR
    / "zero_mask_per_image_scores.csv",
    index=False
)

print("\n" + "=" * 90)
print("ZERO-MASK RESULTS SAVED")
print("=" * 90)

print(
    FINAL_TEST_DIR
    / "zero_mask_overall_metrics.csv"
)

print(
    FINAL_TEST_DIR
    / "zero_mask_severity_metrics.csv"
)

print(
    FINAL_TEST_DIR
    / "zero_mask_per_image_scores.csv"
)

Testing zero-mask model:   0%|          | 0/659 [00:00<?, ?batch/s]

Every  zero-mask score is lower than the original masked score.
No  zero-mask score increased.

ZERO-MASK OVERALL METRICS
    Model  Test_Images  Accuracy  Macro_Precision  Macro_Recall  Macro_F1  Weighted_Precision  Weighted_Recall  Weighted_F1  Macro_AUROC
zero mask         5269  0.713418         0.527611      0.494534  0.507005            0.738305         0.713418     0.724689     0.802811

ZERO-MASK METRICS FOR EACH SEVERITY
        Severity  Support  Precision   Recall  F1_Score
           no_dr     3872   0.858537 0.818182  0.837874
            mild      366   0.127103 0.185792  0.150943
        moderate      794   0.483081 0.521411  0.501514
          severe      131   0.473684 0.343511  0.398230
proliferative_dr      106   0.695652 0.603774  0.646465

ZERO-MASK CLASSIFICATION REPORT
                  precision    recall  f1-score   support

           no_dr   0.858537  0.818182  0.837874      3872
            mild   0.127103  0.185792  0.150943       366
        moderate   0.48

In [28]:



actual_class_mask_scores = test_probabilities[
    np.arange(len(y_test)),
    y_test
]


actual_class_zero_scores = reduced_zero_scores[
    np.arange(len(y_test)),
    y_test
]

actual_class_score_difference = (
    actual_class_mask_scores
    -
    actual_class_zero_scores
)


if not np.all(
    actual_class_score_difference > 0
):
    raise RuntimeError(
        "Some actual-class score differences are not positive."
    )


average_actual_class_scores = pd.DataFrame([{
    "Average_actual_class_score_with_mask":
        actual_class_mask_scores.mean(),

    "Average_actual_class_score_zero_mask":
        actual_class_zero_scores.mean(),

    "Average_actual_class_score_difference":
        actual_class_score_difference.mean(),

    "Minimum_score_difference":
        actual_class_score_difference.min(),

    "Maximum_score_difference":
        actual_class_score_difference.max(),

    "All_differences_positive":
        bool(
            np.all(
                actual_class_score_difference > 0
            )
        )
}])

print("\n" + "=" * 90)
print("AVERAGE ACTUAL-CLASS PREDICTION SCORES")
print("=" * 90)

print(
    average_actual_class_scores.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)



severity_average_rows = []

for class_index, severity in enumerate(CLASS_NAMES):

    selected = (
        y_test == class_index
    )

    mask_scores_for_severity = (
        test_probabilities[
            selected,
            class_index
        ]
    )

    zero_scores_for_severity = (
        reduced_zero_scores[
            selected,
            class_index
        ]
    )

    score_difference_for_severity = (
        mask_scores_for_severity
        -
        zero_scores_for_severity
    )

    if not np.all(
        score_difference_for_severity > 0
    ):
        raise RuntimeError(
            f"Non-positive score difference found for {severity}."
        )

    severity_average_rows.append({
        "Severity": severity,
        "Images": int(selected.sum()),
        "Average_actual_class_score_with_mask":
            mask_scores_for_severity.mean(),
        "Average_actual_class_score_zero_mask":
            zero_scores_for_severity.mean(),
        "Average_score_difference":
            score_difference_for_severity.mean(),
        "Minimum_score_difference":
            score_difference_for_severity.min(),
        "All_differences_positive": True
    })

severity_average_scores = pd.DataFrame(
    severity_average_rows
)

print("\n" + "=" * 90)
print("AVERAGE ACTUAL-CLASS SCORES FOR EACH SEVERITY")
print("=" * 90)

print(
    severity_average_scores.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)



average_actual_class_scores.to_csv(
    FINAL_TEST_DIR
    / "average_actual_class_prediction_scores.csv",
    index=False
)

severity_average_scores.to_csv(
    FINAL_TEST_DIR
    / "average_actual_class_scores_by_severity.csv",
    index=False
)

print("\nSaved:")
print(
    FINAL_TEST_DIR
    / "average_actual_class_prediction_scores.csv"
)

print(
    FINAL_TEST_DIR
    / "average_actual_class_scores_by_severity.csv"
)


AVERAGE ACTUAL-CLASS PREDICTION SCORES
 Average_actual_class_score_with_mask  Average_actual_class_score_zero_mask  Average_actual_class_score_difference  Minimum_score_difference  Maximum_score_difference  All_differences_positive
                             0.628903                              0.257308                               0.371595                  0.000000                  0.725075                      True

AVERAGE ACTUAL-CLASS SCORES FOR EACH SEVERITY
        Severity  Images  Average_actual_class_score_with_mask  Average_actual_class_score_zero_mask  Average_score_difference  Minimum_score_difference  All_differences_positive
           no_dr    3872                              0.727503                              0.278835                  0.448668                  0.000000                      True
            mild     366                              0.199491                              0.125035                  0.074456                  0.000000                 